# SQL query from table names - Continued

In [4]:
!pip install --upgrade openai
!pip install --upgrade python-dotenv


In [5]:
from openai import OpenAI
print("OpenAI client loaded correctly")


OpenAI client loaded correctly


In [16]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(
    api_key=OPENAI_API_KEY
)


## The old Prompt

In [17]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [19]:
context = [
    {'role': 'system', 'content': """
You are an expert SQL generator.
Given a natural language question, generate a valid SQLite query.
Return only the SQL enclosed in triple backticks.
Keep the SQL simple and efficient.
"""},

    {'role': 'system', 'content': """
CREATE TABLE employees (
  ID_usr INTEGER PRIMARY KEY,
  name TEXT
);

INSERT INTO employees VALUES
(1, 'Alice'),
(2, 'Bob'),
(3, 'Charlie');
"""},

    {'role': 'system', 'content': """
CREATE TABLE salary (
  ID_usr INTEGER,
  year DATE,
  salary REAL
);

INSERT INTO salary VALUES
(1, '2023-01-01', 60000),
(2, '2023-01-01', 80000),
(3, '2023-01-01', 50000);
"""},

    {'role': 'system', 'content': """
CREATE TABLE studies (
  ID INTEGER PRIMARY KEY,
  ID_usr INTEGER,
  educational_level INTEGER,
  Institution TEXT,
  Years DATE,
  Speciality TEXT
);

INSERT INTO studies VALUES
(1,1,3,'MIT','2018-01-01','Engineering'),
(2,2,4,'Stanford','2019-01-01','AI'),
(3,3,2,'MIT','2017-01-01','Math');
"""}
]


In [20]:
context.append(
    {'role': 'system', 'content': """
-- Q: Who is the best paid employee?
-- A:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
"""}
)


In [21]:
#Function to call the model.
def return_CCRMSQL(user_message, context):
    messages = context.copy()
    messages.append({
        'role': 'user',
        'content': f"Question: {user_message}"
    })

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=0
    )

    return response.choices[0].message.content


In [22]:
print(
    return_CCRMSQL(
        "Which institution has the highest average salary?",
        context
    )
)


```sql
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN salary sa ON st.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [23]:
print(
    return_CCRMSQL(
        "Who is the best paid employee?",
        context
    )
)



```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```


## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [24]:
context_user = context.copy()

print(
    return_CCRMSQL(
        "Which institution has the highest average salary?",
        context_user
    )
)


```sql
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN employees e ON st.ID_usr = e.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [25]:
old_context_user = old_context.copy()

print(
    return_CCRMSQL(
        "Which institution has the highest average salary?",
        old_context_user
    )
)


This is your SQL:
```sql
SELECT s.Institution, AVG(s.salary) AS avg_salary
FROM salary s
JOIN employees e ON s.ID_usr = e.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query retrieves the institution with the highest average salary by joining the "salary" table with the "employees" table on the user ID. It calculates the average salary for each institution, orders the results in descending order based on the average salary, and then limits the output to only show the institution with the highest average salary.


In [26]:
print(
    return_CCRMSQL(
        "Which institution has the highest average salary?",
        context_user
    )
)


```sql
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN employees e ON st.ID_usr = e.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [27]:
print(
    return_CCRMSQL(
        "Which institution has the highest average salary?",
        old_context_user
    )
)


This is your SQL:
```sql
SELECT s.Institution, AVG(s.salary) AS avg_salary
FROM salary s
JOIN employees e ON s.ID_usr = e.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query retrieves the institution with the highest average salary by joining the `salary` table with the `employees` table on the `ID_usr` column, calculating the average salary for each institution, ordering the results in descending order of average salary, and then limiting the output to only the top result.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?